# Notebook 04 — Positional Encoding

## Learning Objectives
- Understand the order-invariance problem: Transformers treat bags of tokens without PE.
- Use `SinusoidalPositionalEncoding` and `LearnedPositionalEncoding` from `src`.
- Visualize the PE heatmap to see the wave patterns across positions and dimensions.
- Measure cosine similarity between different positions to see how PE encodes distance.
- Compare sinusoidal (fixed) vs learned PE: when to use each.

## Big Picture
"**dog bites man**" and "**man bites dog**" have exactly the same token set but opposite meanings.
Without positional encoding, a Transformer cannot distinguish them.
Positional encodings inject position information by adding a unique vector to each token embedding.

## 1. Imports and Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_markdown_report
from src.utils.paths import RUNS_ATTENTION_DIR
from src.models.positional_encoding import (
    SinusoidalPositionalEncoding,
    LearnedPositionalEncoding,
)

seed_everything(42)
device = get_best_device()
prepare_run_dir(RUNS_ATTENTION_DIR, clean=False)
print(f"Device: {device}")
print("All imports successful.")

## 2. Dataset: Order-Sensitive Sentence Pairs

We use toy sentences to demonstrate the order-invariance problem:
- **"dog bites man"** — Dog is the agent, man is the patient.
- **"man bites dog"** — Man is the agent, dog is the patient.

Same tokens, opposite meaning. Without PE, a Transformer cannot tell these apart.

In [ ]:
# Two sentences with identical token sets but different order
sent1 = ["dog", "bites", "man"]
sent2 = ["man", "bites", "dog"]

# Token IDs (pretend we have a 3-word vocabulary: dog=0, bites=1, man=2)
vocab = {"dog": 0, "bites": 1, "man": 2}
ids1 = torch.tensor([[vocab[w] for w in sent1]])  # [1, 3]
ids2 = torch.tensor([[vocab[w] for w in sent2]])  # [1, 3]

print(f"Sentence 1: {sent1}  → ids: {ids1.tolist()}")
print(f"Sentence 2: {sent2}  → ids: {ids2.tolist()}")

# Create simple learnable embeddings for demo
D_MODEL = 16
torch.manual_seed(42)
embed = torch.nn.Embedding(3, D_MODEL)  # 3 words, D_MODEL-dim embeddings

emb1 = embed(ids1)   # [1, 3, D_MODEL]
emb2 = embed(ids2)   # [1, 3, D_MODEL]

print(f"\nEmbedding of 'dog bites man' shape: {emb1.shape}")

# Without PE: sort the embeddings to show they look the same set-wise
sorted_emb1 = emb1.squeeze(0).sort(dim=0).values
sorted_emb2 = emb2.squeeze(0).sort(dim=0).values
print(f"\nWithout PE, sorted embeddings identical? {torch.allclose(sorted_emb1, sorted_emb2)}")
print("(Both sentences share the same token set → same sorted embedding bag)")

## 3. Architecture (Text Diagram)

```
Input token IDs:  [dog, bites, man]   (= [0, 1, 2])
        │
  Token Embedding:   [E_dog, E_bites, E_man]   shape [1, 3, d_model]
        │
  + Positional Encoding:
        PE(pos=0) = [sin(0/10000^0), cos(0/10000^0), ...]
        PE(pos=1) = [sin(1/10000^0), cos(1/10000^0), ...]
        PE(pos=2) = [sin(2/10000^0), cos(2/10000^0), ...]
        │
  Result: E_dog + PE(0),  E_bites + PE(1),  E_man + PE(2)

Now 'dog bites man' ≠ 'man bites dog' even though they share the same tokens.
```

## 4. Theory: Sinusoidal Positional Encoding

The original Transformer paper uses fixed sinusoidal encodings:

$$PE(pos, 2i) = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

**Key properties**:
1. Each position gets a unique encoding (no two rows are the same).
2. Nearby positions have similar encodings (smooth interpolation).
3. The dot product of PE(pos) and PE(pos+k) depends only on k, not the absolute position.
4. The model can generalize to longer sequences than seen at training time.

**Learned PE** simply assigns a learnable vector to each position — simpler but
cannot generalize beyond max_len.

## 5. Implementation: Comparing the Two Encodings

In [ ]:
D_MODEL  = 64
MAX_LEN  = 50   # positions to visualize
SEQ_LEN  = 6    # for demo sentences

# ── Sinusoidal PE ──
sin_pe = SinusoidalPositionalEncoding(d_model=D_MODEL, max_len=MAX_LEN, dropout=0.0)
sin_pe.eval()

# Extract the PE matrix directly from the buffer
pe_matrix = sin_pe.pe[0, :MAX_LEN, :].detach().numpy()  # [MAX_LEN, D_MODEL]
print(f"Sinusoidal PE matrix shape: {pe_matrix.shape}  (max_len, d_model)")
print(f"PE values range: [{pe_matrix.min():.3f}, {pe_matrix.max():.3f}]  (should be in [-1, 1])")

# ── Learned PE ──
torch.manual_seed(42)
learned_pe = LearnedPositionalEncoding(d_model=D_MODEL, max_len=MAX_LEN, dropout=0.0)
learned_pe.eval()

# Extract embeddings for all positions
positions = torch.arange(MAX_LEN).unsqueeze(0)  # [1, MAX_LEN]
learned_matrix = learned_pe.position_embedding(positions)[0].detach().numpy()  # [MAX_LEN, D_MODEL]
print(f"\nLearned PE matrix shape: {learned_matrix.shape}  (max_len, d_model)")

In [ ]:
# Demonstrate that PE makes 'dog bites man' ≠ 'man bites dog'
emb_dim = D_MODEL
embed64 = torch.nn.Embedding(3, emb_dim)
torch.manual_seed(42)
torch.nn.init.normal_(embed64.weight, mean=0, std=0.1)

sin_pe64 = SinusoidalPositionalEncoding(d_model=emb_dim, max_len=10, dropout=0.0)
sin_pe64.eval()

# Without PE
e1_no_pe = embed64(ids1)   # [1, 3, 64]
e2_no_pe = embed64(ids2)   # [1, 3, 64]

# With PE
with torch.no_grad():
    e1_with_pe = sin_pe64(embed64(ids1))  # [1, 3, 64]
    e2_with_pe = sin_pe64(embed64(ids2))  # [1, 3, 64]

diff_no_pe   = (e1_no_pe   - e2_no_pe).abs().mean().item()
diff_with_pe = (e1_with_pe - e2_with_pe).abs().mean().item()

print("Order sensitivity test:")
print(f"  'dog bites man' vs 'man bites dog'")
print(f"  Mean absolute diff WITHOUT PE : {diff_no_pe:.6f}")
print(f"  Mean absolute diff WITH    PE : {diff_with_pe:.6f}")
print(f"  PE makes sentences distinguishable: {diff_with_pe > diff_no_pe}")

In [ ]:
# Cosine similarity between positions
def cosine_similarity_matrix(matrix: np.ndarray) -> np.ndarray:
    """Compute pairwise cosine similarity matrix. matrix: [n, d]"""
    norms = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-9
    normalized = matrix / norms
    return normalized @ normalized.T

N_SHOW = 20  # show first 20 positions
sin_sim  = cosine_similarity_matrix(pe_matrix[:N_SHOW])
print(f"Position cosine similarity matrix shape: {sin_sim.shape}")
print(f"Similarity between pos 0 and pos 1: {sin_sim[0, 1]:.4f}")
print(f"Similarity between pos 0 and pos 10: {sin_sim[0, 10]:.4f}")
print(f"Similarity between pos 0 and pos 19: {sin_sim[0, 19]:.4f}")
print("(Nearby positions should be more similar than distant ones)")

## 6. Sanity Checks

In [ ]:
# Shape checks
assert pe_matrix.shape == (MAX_LEN, D_MODEL), f"{pe_matrix.shape}"
assert learned_matrix.shape == (MAX_LEN, D_MODEL), f"{learned_matrix.shape}"
print("Shape checks passed.")

# Sinusoidal PE values should be in [-1, 1]
assert pe_matrix.min() >= -1.01 and pe_matrix.max() <= 1.01, \
    f"Sinusoidal PE out of range: [{pe_matrix.min():.3f}, {pe_matrix.max():.3f}]"
print("Sinusoidal PE value range check passed.")

# PE forward pass should not change sequence shape
dummy = torch.randn(2, MAX_LEN, D_MODEL)
with torch.no_grad():
    out = sin_pe(dummy)
assert out.shape == dummy.shape, f"PE changed shape: {out.shape}"
print(f"PE preserves input shape: {dummy.shape} → {out.shape}")

# With PE, the two sentences should have different representations
assert diff_with_pe > 0, "PE made no difference!"
print("PE distinguishes word-order pairs. All checks passed!")

## 7. Visualization

In [ ]:
# Plot 1: PE heatmap showing wave patterns across positions and dimensions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sinusoidal
ax = axes[0]
im = ax.imshow(pe_matrix.T, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax.set_xlabel('Position')
ax.set_ylabel('Embedding dimension')
ax.set_title('Sinusoidal PE\n(rows = dimensions, cols = positions)')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Learned (random init)
ax = axes[1]
im = ax.imshow(learned_matrix.T, cmap='RdBu_r', aspect='auto')
ax.set_xlabel('Position')
ax.set_ylabel('Embedding dimension')
ax.set_title('Learned PE (random init)\n(rows = dimensions, cols = positions)')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('Positional Encoding Heatmaps', fontsize=13)
fig.tight_layout()

pe_path = RUNS_ATTENTION_DIR / 'positional_encoding_heatmap.png'
fig.savefig(pe_path, dpi=120)
plt.close(fig)
print(f"PE heatmap saved to: {pe_path}")

In [ ]:
# Plot 2: Cosine similarity between positions (shows how PE encodes distance)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Heatmap of pairwise similarities
ax = axes[0]
im = ax.imshow(sin_sim, cmap='viridis', vmin=-1, vmax=1)
ax.set_title(f'Pairwise PE Cosine Similarity\n(first {N_SHOW} positions)')
ax.set_xlabel('Position j')
ax.set_ylabel('Position i')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Line plot: similarity of pos 0 vs all other positions
ax = axes[1]
ax.plot(range(N_SHOW), sin_sim[0], 'o-', color='steelblue', label='PE(0) vs PE(j)')
ax.plot(range(N_SHOW), sin_sim[5], 's-', color='tomato',    label='PE(5) vs PE(j)')
ax.plot(range(N_SHOW), sin_sim[10], '^-', color='green',   label='PE(10) vs PE(j)')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Position j')
ax.set_ylabel('Cosine Similarity')
ax.set_title('Similarity Decays with Distance')
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
sim_path = RUNS_ATTENTION_DIR / 'position_similarity.png'
fig.savefig(sim_path, dpi=120)
plt.close(fig)
print(f"Position similarity plot saved to: {sim_path}")

In [ ]:
# Save session report
report_path = RUNS_ATTENTION_DIR / 'session_report.md'
save_markdown_report(
    title='Positional Encoding',
    summary=(
        'Demonstrated that sinusoidal PE makes word-order distinguishable. '
        'Visualized PE heatmap and position-to-position cosine similarities.'
    ),
    metrics={
        'd_model': D_MODEL,
        'max_len': MAX_LEN,
        'pe_value_min': float(pe_matrix.min()),
        'pe_value_max': float(pe_matrix.max()),
        'diff_no_pe':   float(diff_no_pe),
        'diff_with_pe': float(diff_with_pe),
    },
    figures=['positional_encoding_heatmap.png', 'position_similarity.png'],
    tables=[],
    output_path=report_path,
    device=str(device),
    hyperparams={'d_model': D_MODEL, 'max_len': MAX_LEN},
    architecture='SinusoidalPositionalEncoding + LearnedPositionalEncoding',
)
print(f"Session report saved to: {report_path}")

## 8. Failure Cases

**Forgetting to add PE**: Without positional encoding, "dog bites man" and "man bites dog" have identical mean-pooled representations → the model cannot classify them differently.

**Learned PE beyond max_len**: `LearnedPositionalEncoding` throws an index error if you feed a sequence longer than max_len. Sinusoidal PE doesn't have this problem.

**PE vs absolute position**: Sinusoidal PE encodes absolute position. For tasks where relative position matters more (translation), newer techniques like Rotary Position Embedding (RoPE) are preferred.

**Scaling after adding PE**: Token embeddings are scaled by √d_model before adding PE (see `text_transformer.py`). If you skip this scale, token information drowns out PE.

## 9. Exercises

1. **Wave frequencies**: Plot individual PE dimensions (dim 0, 2, 4, ...) as a function of position. Notice how lower dimensions have lower frequency (longer wavelength).
2. **d_model effect**: Plot PE heatmaps for d_model=16, 64, 256. How does increasing d_model change the heatmap texture?
3. **Relative position property**: Show that PE(pos) · PE(pos+k) is approximately constant for different `pos` values with the same `k`. (This is a key property of sinusoidal PE.)
4. **Learned PE after training**: Train a small Transformer on a sequence classification task for 5 epochs. Then visualize the learned PE matrix. Does it look more structured than random?
5. **No PE ablation**: Build a tiny Transformer with and without PE. Train both on "dog bites man" vs "man bites dog" classification. Which converges? Why?

## 10. Key Takeaways

- Transformers are **order-invariant without PE** — "dog bites man" = "man bites dog" as token sets.
- **Sinusoidal PE** uses sin/cos at different frequencies to give each position a unique fingerprint.
- **Learned PE** is simpler but cannot generalize past max_len; sinusoidal PE can.
- **Cosine similarity decays** with positional distance — nearby positions are more similar.
- PE is added to token embeddings before the first encoder block, *not* inside the attention module.